# Kappa-scale calibration on the Colab T4

Calibrates the single free scale parameter of a heterogeneous
capital-productivity Krusell-Smith economy: the cross-sectional *shape* of
`kappa` is hardcoded from Xavier (2021)'s return-on-wealth-by-percentile
curve (already cited in the paper), and `k_multiplier` is swept over a grid,
scoring how closely each trained economy's own emergent steady-state return
distribution reproduces that same curve. Full design writeup:
`runs/ks-heterogeneous-returns/README.md`.

Each grid point is a full training run on `configs/exp/ks_n200.yaml` (the
paper's own validated n=200 protocol, ~3 min/cell on a free T4); the default
7-point grid is one Colab session.


In [ ]:
# Setup: clone or update the repo, install (idempotent -- safe to re-run).
%cd /content
![ -d jax-marl-bc ] || git clone https://github.com/danmonuni/jax-marl-bc.git
%cd jax-marl-bc
!git pull
!pip install -q -r requirements.txt && pip install -q -e . --no-deps


In [ ]:
# Sanity: a GPU runtime is attached (Runtime > Change runtime type > T4 GPU).
!nvidia-smi -L


In [ ]:
# Mount Drive BEFORE the run so the result is saved as soon as it finishes
# (a Colab disconnect then loses at most the run in progress, never a
# finished one).
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

def save_results(name='ks-heterogeneous-returns'):
    """Sync runs/<name>/results -> Drive (exact path, idempotent re-sync)."""
    src = f'runs/{name}/results'
    dst = f'/content/drive/MyDrive/jax-marl-bc-runs/{name}/results'
    assert os.path.exists(src), f"{src} missing - did the calibration run finish?"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"saved {src} -> {dst}")


## Calibration grid

Runs `runs/ks-heterogeneous-returns/config.yaml` as-is: `n_agents=200`,
`device=gpu`, and the default `k_grid` (0.1-0.5, centered on the naive
`1/mean(base_vector) ~ 0.26` guess). Pass dotlist overrides after the script
path to change any of these, e.g. a narrower follow-up grid around the best
point from a first pass:
`!python runs/ks-heterogeneous-returns/calibrate_kappa_scale.py "k_grid=[0.22,0.24,0.26,0.28]"`


In [ ]:
!python runs/ks-heterogeneous-returns/calibrate_kappa_scale.py


In [ ]:
save_results('ks-heterogeneous-returns')


## Results


In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('runs/ks-heterogeneous-returns/results/results.csv')
display(df[['k_multiplier', 'score_rms_ratio_minus_1', 'capital_gini', 'top_0.1_share']])

fig_path = 'runs/ks-heterogeneous-returns/results/calibration_fit.png'
display(Image(fig_path))
